# ML Assignment 2 – Breast Cancer Wisconsin (Diagnostic)
Train and evaluate 5 classifiers (Logistic Regression, Decision Tree, KNN, Naive Bayes, SVM).

**Run the cells one by one from top to bottom** with Shift + Enter.


## 1. Environment details (keep these for Streamlit Cloud)

In [ ]:
import sys, platform, sklearn
print("Python      :", platform.python_version())
print("scikit-learn:", sklearn.__version__)

## 2. Get the project files
**Recommended:** paste your GitHub repository link between the quotes of `REPO_URL` and run the cell – the repo is cloned.

If `REPO_URL` is left empty, a **Choose Files** button appears instead – upload `bc-classifier-app.zip`.

In [ ]:
import os, glob, subprocess
os.chdir("/content")

REPO_URL = ""   # e.g. "https://github.com/<your-username>/<your-repo-name>"

if REPO_URL.strip():
    repo = REPO_URL.strip().rstrip("/").removesuffix(".git")
    PROJECT = "/content/" + repo.split("/")[-1]
    subprocess.run(["rm", "-rf", PROJECT])
    subprocess.run(["git", "clone", repo + ".git", PROJECT], check=True)
else:
    from google.colab import files
    uploaded = files.upload()                 # choose bc-classifier-app.zip
    zip_name = list(uploaded.keys())[0]
    subprocess.run(["unzip", "-o", "-q", zip_name], check=True)
    PROJECT = "/content/bc-classifier-app"

os.chdir(PROJECT)
print("Project folder:", PROJECT)
!ls -la

## 3. Install the extra package (UCI dataset downloader)

In [ ]:
!pip install -q ucimlrepo

## 4. Train, tune and evaluate all 5 models
Loads data from the **UCI repository online**; if the internet fails it falls back to the local copy `data/wdbc.csv` automatically.
To force offline mode use `!python train_models.py --offline`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
!python -W ignore train_models.py

## 5. Comparison table and confusion matrices

In [ ]:
import json, joblib, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import confusion_matrix
from data_utils import FEATURE_COLUMNS, TARGET_COLUMN, encode_target

display(pd.read_csv("results/metrics_comparison.csv", index_col=0))

meta = json.load(open("model/metadata.json"))
test = pd.read_csv("test_data.csv")
X, y = test[FEATURE_COLUMNS], encode_target(test[TARGET_COLUMN])

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (name, fname) in zip(axes, meta["models"].items()):
    model = joblib.load(f"model/{fname}")
    cm = confusion_matrix(y, model.predict(X), labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cbar=False, cmap="crest", ax=ax,
                xticklabels=["Benign", "Malignant"], yticklabels=["Benign", "Malignant"])
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.show()

## 6. Classification reports

In [ ]:
print(open('results/classification_reports.txt').read())

## 7. Download the trained project (optional)
Skip this cell if you only ran the notebook for the BITS Virtual Lab screenshot – the files for GitHub come from your laptop.


In [ ]:
os.chdir("/content")
folder = os.path.basename(PROJECT)
!rm -rf {folder}/__pycache__
!zip -q -r bc-classifier-app-trained.zip {folder} -x "*.ipynb_checkpoints*" "*/.git/*"
from google.colab import files
files.download("bc-classifier-app-trained.zip")
